In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.stats.power import TTestIndPower

In [ ]:
PILOT_CSV = r"C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results\pilot_20260424_001142.csv"
RESULTS_DIR = Path(r"C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results")

In [ ]:
ENERGY_COL = "joules"
GROUP_COLS = ["language", "algorithm", "size"]
 

In [ ]:
N_LANGUAGES       = 6
N_COMPARISONS     = (N_LANGUAGES * (N_LANGUAGES - 1)) // 2   # = 15
ALPHA_NOMINAL     = 0.05
ALPHA_CORRECTED   = ALPHA_NOMINAL / N_COMPARISONS             # Bonferroni
EFFECT_SIZE_D     = 0.5                                        # medium (Cohen 1988)
POWER_TARGET      = 0.80                                       # 80% power

In [ ]:
# 1. Load pilot data

df = pd.read_csv(PILOT_CSV)
print(f"Loaded {len(df)} rows from pilot CSV.")
print(f"Columns: {list(df.columns)}\n")

In [ ]:
# 2. Compute per-cell statistics
# --------------------------------------------------
stats = df.groupby(GROUP_COLS)[ENERGY_COL].agg(
    count="count",
    mean="mean",
    std="std"
).reset_index()
stats["cv_pct"] = (stats["std"] / stats["mean"]) * 100

In [ ]:
# 3. Power analysis
    # --------------------------------------------------
    # TTestIndPower solves for sample size per group given:
    #   effect_size : Cohen's d
    #   alpha       : Bonferroni-corrected significance level
    #   power       : desired statistical power
    #   alternative : 'two-sided'
analysis = TTestIndPower()
n_required = analysis.solve_power(
    effect_size=EFFECT_SIZE_D,
    alpha=ALPHA_CORRECTED,
    power=POWER_TARGET,
    alternative="two-sided"
)
n_required_ceil = int(np.ceil(n_required))
 

In [ ]:
separator = "=" * 65

In [ ]:
print(separator)
print("POWER ANALYSIS PARAMETERS")
print(separator)
print(f"  Number of languages          : {N_LANGUAGES}")
print(f"  Pairwise comparisons (m)     : {N_COMPARISONS}")
print(f"  Nominal alpha                : {ALPHA_NOMINAL}")
print(f"  Bonferroni-corrected alpha'  : {ALPHA_CORRECTED:.6f}  (= {ALPHA_NOMINAL}/{N_COMPARISONS})")
print(f"  Target effect size (Cohen's d): {EFFECT_SIZE_D}  (medium)")
print(f"  Target power (1 - beta)      : {POWER_TARGET}")
print(f"  Test type                    : Two-sided Welch's t-test")
 

In [ ]:
print(f"\n{separator}")
print("RESULT")
print(separator)
print(f"  Raw N from solver            : {n_required:.4f}")
print(f"  Recommended N per cell       : {n_required_ceil}  (ceiling)")

In [ ]:
print(f"\n{separator}")
print("PILOT STUDY VARIANCE SUMMARY")
print(separator)
print(f"  CV range   : {stats['cv_pct'].min():.2f}% -- {stats['cv_pct'].max():.2f}%")
print(f"  Mean CV    : {stats['cv_pct'].mean():.2f}%")
print(f"  Median CV  : {stats['cv_pct'].median():.2f}%")

In [ ]:
print(f"\n  Per-language mean CV:")
lang_cv = stats.groupby("language")["cv_pct"].mean().sort_values()
for lang, cv in lang_cv.items():
    print(f"    {lang:<12} : {cv:.2f}%")

In [ ]:
# 5. Save results to file
# --------------------------------------------------
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = RESULTS_DIR / "power_analysis_results.txt"
with open(out_path, "w") as f:
    f.write("POWER ANALYSIS RESULTS\n")
    f.write(separator + "\n\n")
    f.write(f"Pilot CSV            : {PILOT_CSV}\n")
    f.write(f"N languages          : {N_LANGUAGES}\n")
    f.write(f"N comparisons (m)    : {N_COMPARISONS}\n")
    f.write(f"Alpha nominal        : {ALPHA_NOMINAL}\n")
    f.write(f"Alpha corrected      : {ALPHA_CORRECTED:.6f}\n")
    f.write(f"Effect size (d)      : {EFFECT_SIZE_D}\n")
    f.write(f"Power target         : {POWER_TARGET}\n")
    f.write(f"Raw N from solver    : {n_required:.4f}\n")
    f.write(f"Recommended N        : {n_required_ceil}\n\n")
    f.write("PAPER-READY SUMMARY\n")
    f.write(separator + "\n")
    f.write(summary + "\n")

print(f"\nResults saved to: {out_path}")